In [1]:
!pip install -q crewai crewai-tools pymupdf python-docx

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
import os
import fitz
import zipfile

from pathlib import Path
from getpass import getpass
from docx import Document
from google.colab import files

from crewai import Agent, Task, Crew, Process


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
print("AI RESUME ANALYZER")
print("==================")

openai_key = getpass("Enter OpenAI API Key: ").strip()

if not openai_key:
    raise ValueError("OpenAI API Key is required.")

os.environ["OPENAI_API_KEY"] = openai_key
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

serper_key = getpass(
    "Enter Serper API Key (press Enter to skip): "
).strip()

search_tools = []

if serper_key:
    os.environ["SERPER_API_KEY"] = serper_key
    from crewai_tools import SerperDevTool
    search_tools = [SerperDevTool()]

print("\nAPI configuration completed.")

AI RESUME ANALYZER
Enter OpenAI API Key: ··········
Enter Serper API Key (press Enter to skip): ··········

API configuration completed.


In [4]:
def extract_text_from_pdf(file_path):

    doc = fitz.open(file_path)

    text = ""

    for page in doc:
        text += page.get_text() + "\n"

    doc.close()

    return text

In [5]:
def extract_text_from_docx(file_path):

    doc = Document(file_path)

    full_text = []

    for para in doc.paragraphs:
        full_text.append(para.text)

    for table in doc.tables:
        for row in table.rows:
            full_text.append(
                " | ".join(cell.text for cell in row.cells)
            )

    return "\n".join(full_text)


In [6]:
def extract_text_from_resume(file_path):

    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return extract_text_from_pdf(file_path)

    elif extension == ".docx":
        return extract_text_from_docx(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Upload PDF or DOCX."
        )

In [7]:
print("\nUpload your Resume (PDF or DOCX).")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No resume uploaded.")

resume_path = next(iter(uploaded.keys()))

resume_text = extract_text_from_resume(resume_path)

if not resume_text.strip():
    raise ValueError(
        "No text found in the resume. Check your file."
    )

print("\nResume uploaded successfully!")
print("Resume text extracted successfully!")


Upload your Resume (PDF or DOCX).


Saving DarshanaSonarResume.pdf to DarshanaSonarResume.pdf

Resume uploaded successfully!
Resume text extracted successfully!


In [8]:
target_role = input(
    "\nEnter your target job role: "
).strip()

location = input(
    "Enter your preferred job location: "
).strip()

if not target_role:
    target_role = "Data Analyst"

if not location:
    location = "India"


Enter your target job role: Data Scientist
Enter your preferred job location: Banglore


In [10]:
# STEP 9: AGENT 1 - RESUME STRATEGIST

resume_feedback = Agent(
    role="Professional Resume Advisor",

    goal=(
        "Analyze resumes and provide detailed "
        "professional feedback."
    ),

    backstory=(
        "You are an experienced professional resume advisor. "
        "You identify strengths, weaknesses, missing skills, "
        "formatting issues and improvements in resumes."
    ),

    verbose=True,

    allow_delegation=False,

    llm="openai/gpt-4o-mini"
)

In [20]:
resume_feedback_task = Task(

    description=f"""
    Analyze the candidate's resume carefully.

    Review every section:

    1. Professional Summary
    2. Education
    3. Technical Skills
    4. Projects
    5. Internship Experience
    6. Achievements
    7. Certifications
    8. Grammar and Formatting
    9. ATS Compatibility

    Identify strengths and weaknesses.

    Suggest relevant improvements.

    Give an overall resume score out of 10.

    Do not invent qualifications or experience.

    RESUME:

    {resume_text}
    """,

    expected_output="""
    A detailed resume evaluation report containing:

    1. Overall resume score out of 10
    2. Resume strengths
    3. Resume weaknesses
    4. Missing skills
    5. ATS compatibility feedback
    6. Improvement suggestions
    """,

    agent=resume_feedback,

    output_file="resume_feedback.md"
)

In [21]:
resume_advisor = Agent(

    role="Professional Resume Writer",

    goal=(
        "Improve the candidate's resume using "
        "professional resume writing techniques."
    ),

    backstory=(
        "You are an expert resume writer. "
        "You create professional, clear and ATS-friendly resumes. "
        "You improve the presentation of skills, projects "
        "and education without adding false information."
    ),

    verbose=True,

    allow_delegation=False,

    llm="openai/gpt-4o-mini"
)

In [22]:
resume_advisor_task = Task(

    description=f"""
    Rewrite and improve the resume using the feedback
    provided by the Resume Advisor.

    Improve these sections:

    1. Professional Summary
    2. Technical Skills
    3. Projects
    4. Education
    5. Internship Experience
    6. Achievements
    7. Grammar and Formatting

    Target Job Role: {target_role}

    Requirements:

    - Use professional language.
    - Make the resume ATS-friendly.
    - Highlight relevant skills.
    - Improve project descriptions.
    - Keep the resume easy to read.
    - Do not create fake experience.
    - Do not invent qualifications or achievements.

    ORIGINAL RESUME:

    {resume_text}
    """,

    expected_output="""
    A complete improved resume in Markdown format.

    Include professional headings, summary, skills,
    education, projects and experience where available.
    """,

    context=[resume_feedback_task],

    agent=resume_advisor,

    output_file="improved_resume.md"
)


In [23]:
job_advisor = Agent(

    role="Career and Job Recommendation Specialist",

    goal=(
        "Recommend suitable career opportunities "
        "based on the candidate's resume and preferences."
    ),

    backstory=(
        "You are an experienced career advisor. "
        "You understand job requirements, technical skills, "
        "entry-level opportunities and career development."
    ),

    tools=search_tools,

    verbose=True,

    allow_delegation=False,

    llm="openai/gpt-4o-mini"
)


In [24]:
if search_tools:

    job_description = f"""
    Find relevant current job opportunities.

    Target Role: {target_role}

    Preferred Location: {location}

    Search for fresher and entry-level opportunities.

    Include:

    1. Company Name
    2. Job Title
    3. Required Skills
    4. Job Location
    5. Application Link

    Use web search to verify opportunities.

    Do not invent job vacancies or application links.

    Candidate Resume:

    {resume_text}
    """

    job_expected_output = """
    A report containing relevant current job opportunities,
    company names, job titles, skills, locations
    and verified source links.
    """

else:

    job_description = f"""
    Recommend suitable career roles based on the resume.

    Target Role: {target_role}

    Preferred Location: {location}

    Include:

    1. Five suitable job roles
    2. Required technical skills
    3. Skills to improve
    4. Career development suggestions
    5. Suitable entry-level career paths

    Do not claim these are verified current vacancies.

    Candidate Resume:

    {resume_text}
    """

    job_expected_output = """
    Five suitable career recommendations,
    required technical skills and a career improvement plan.
    """

job_advisor_task = Task(

    description=job_description,

    expected_output=job_expected_output,

    context=[
        resume_feedback_task,
        resume_advisor_task
    ],

    agent=job_advisor,

    output_file="job_recommendations.md"
)

In [25]:
resume_crew = Crew(

    agents=[
        resume_feedback,
        resume_advisor,
        job_advisor
    ],

    tasks=[
        resume_feedback_task,
        resume_advisor_task,
        job_advisor_task
    ],

    process=Process.sequential,

    verbose=True
)

In [27]:

# STEP 16: EXECUTE THE AI RESUME ANALYZER

print("\n====================================")
print("STARTING AI RESUME ANALYZER")
print("====================================")

try:
    result = await resume_crew.kickoff_async()

    print("\n====================================")
    print("AI RESUME ANALYSIS COMPLETED SUCCESSFULLY")
    print("====================================")

    print("\nFINAL ANALYSIS:")
    print(result)

except Exception as e:
    print("\n====================================")
    print("ERROR DURING CREW EXECUTION")
    print("====================================")
    print(type(e).__name__, ":", str(e))



STARTING AI RESUME ANALYZER


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2978e3f0-89d5-4708-b401-844f251d15db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the candidate's resume carefully.                                                                  │
│                                                                                                                 │
│      Review every section:                                                                                      │
│                                                                                                                 │
│      1. Professional Summary                                                                                    │
│      2. Education                                                                                               │
│      3. Technical Skills                                                                                        │
│      4. Projects                                                                                                │
│      5. Internship Experience                                                                                   │
│      6. Achievements                                                                                            │
│      7. Certifications                                                                                          │
│      8. Grammar and Formatting                                                                                  │
│      9. ATS Compatibility                                                                                       │
│                                                                                                                 │
│      Identify strengths and weaknesses.                                                                         │
│                                                                                                                 │
│      Suggest relevant improvements.                                                                             │
│                                                                                                                 │
│      Give an overall resume score out of 10.                                                                    │
│                                                                                                                 │
│      Do not invent qualifications or experience.                                                                │
│                                                                                                                 │
│      RESUME:                                                                                                    │
│                                                                                                                 │
│      D A R S H N A S O N A R                                                                                    │
│  7666560071  |  sonardarshna7@gmail.com  |  Nashik 422001                                                       │
│  https://www.linkedin.com/in/darshna-sonar-20b594340?                                                           │
│  OBJECTIVE                                                                                                      │
│  Ambitious B.Tech Artificial Intelligence & Data Science student with hands-on project experience               │
│  in end-to-end machine learning workflows, predictive analytics, and NLP solutions. Seeking to leverage         │
│  strong Python programming capabilities, statistical pr

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Professional Resume Advisor                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the candidate's resume carefully.                                                                  │
│                                                                                                                 │
│      Review every section:                                                                                      │
│                                                                                                                 │
│      1. Professional Summary                                                                                    │
│      2. Education                                                                                               │
│      3. Technical Skills                                                                                        │
│      4. Projects                                                                                                │
│      5. Internship Experience                                                                                   │
│      6. Achievements                                                                                            │
│      7. Certifications                                                                                          │
│      8. Grammar and Formatting                                                                                  │
│      9. ATS Compatibility                                                                                       │
│                                                                                                                 │
│      Identify strengths and weaknesses.                                                                         │
│                                                                                                                 │
│      Suggest relevant improvements.                                                                             │
│                                                                                                                 │
│      Give an overall resume score out of 10.                                                                    │
│                                                                                                                 │
│      Do not invent qualifications or experience.                                                                │
│                                                                                                                 │
│      RESUME:                                                                                                    │
│                                                                                                                 │
│      D A R S H N A S O N A R                                                                                    │
│  7666560071  |  sonardarshna7@gmail.com  |  Nashik 422001                                                       │
│  https://www.linkedin.com/in/darshna-sonar-20b594340?                                                           │
│  OBJECTIVE                                                                                                      │
│  Ambitious B.Tech Artificial Intelligence & Data Science student with hands-on project experience               │
│  in end-to-end machine learning workflows, predictive a

ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************3HZg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
ERROR:root:OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************3HZg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided:            │
│  AQ.Ab8RN*****************************************3HZg. You can find your API key at                            │
│  https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code':        │
│  'invalid_api_key'}}                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (4 outputs left out)

An unknown error occurred. Please check the details below.
Error details: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************3HZg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 2978e3f0-89d5-4708-b401-844f251d15db                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name:                                                                                                          │
│      Analyze the candidate's resume carefully.                                                                  │
│                                                                                                                 │
│      Review every section:                                                                                      │
│                                                                                                                 │
│      1. Professional Summary                                                                                    │
│      2. Education                                                                                               │
│      3. Technical Skills                                                                                        │
│      4. Projects                                                                                                │
│      5. Internship Experience                                                                                   │
│      6. Achievements                                                                                            │
│      7. Certifications                                                                                          │
│      8. Grammar and Formatting                                                                                  │
│      9. ATS Compatibility                                                                                       │
│                                                                                                                 │
│      Identify strengths and weaknesses.                                                                         │
│                                                                                                                 │
│      Suggest relevant improvements.                                                                             │
│                                                                                                                 │
│      Give an overall resume score out of 10.                                                                    │
│                                                                                                                 │
│      Do not invent qualifications or experience.                                                                │
│                                                                                                                 │
│      RESUME:                                                                                                    │
│                                                                                                                 │
│      D A R S H N A S O N A R                                                                                    │
│  7666560071  |  sonardarshna7@gmail.com  |  Nashik 422001                                                       │
│  https://www.linkedin.com/in/darshna-sonar-20b594340?                                                           │
│  OBJECTIVE                                                                                                      │
│  Ambitious B.Tech Artificial Intelligence & Data Science student with hands-on project experience               │
│  in end-to-end machine learning workflows, predictive analytics, and NLP solutions. Seeking to leverage         │
│  strong Python programming capabilities, statistical pr


ERROR DURING CREW EXECUTION
AuthenticationError : Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************3HZg. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}


In [28]:
output_files = [
    ("resume_feedback.md", resume_feedback_task),
    ("improved_resume.md", resume_advisor_task),
    ("job_recommendations.md", job_advisor_task)
]

for filename, task in output_files:

    if not os.path.exists(filename):

        if task.output:
            with open(filename, "w", encoding="utf-8") as file:
                file.write(str(task.output.raw))

In [32]:
zip_filename = "AI_Resume_Analyzer_Results.zip"

with zipfile.ZipFile(
    zip_filename, "w", zipfile.ZIP_DEFLATED
) as zip_file:

    for filename, task in output_files:

        if os.path.exists(filename):
            zip_file.write(filename)

print("\n====================================")
print("PROJECT OUTPUT FILES")
print("====================================")

for filename, task in output_files:

    if os.path.exists(filename):
        print("Created:", filename)

print("\nAll analysis reports are ready.")

# STEP 19: DOWNLOAD RESULTS

files.download(zip_filename)

print("\nAI RESUME ANALYZER PROJECT COMPLETED!")


PROJECT OUTPUT FILES

All analysis reports are ready.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


AI RESUME ANALYZER PROJECT COMPLETED!
